In [1]:
import sys
from pathlib import Path

print("Python version:")
print(sys.version)

print("\nPython executable:")
print(sys.executable)

print("\nCurrent working directory:")
print(Path.cwd())

Python version:
3.9.2 (tags/v3.9.2:1a79785, Feb 19 2021, 13:44:55) [MSC v.1928 64 bit (AMD64)]

Python executable:
c:\Users\sa\AppData\Local\Programs\Python\Python39\python.exe

Current working directory:
c:\Users\sa\Desktop\MLOps-Qafza-2026\Tasks\Task-02


In [2]:
import pandas as pd
import numpy as np
import sklearn

ARTIFACT_DIR = Path("artifacts")

data_paths = {
    "train": ARTIFACT_DIR / "train.csv",
    "validation": ARTIFACT_DIR / "val.csv",
    "test": ARTIFACT_DIR / "test.csv",
}

# Ensure that all required files exist
for name, path in data_paths.items():
    if not path.exists():
        raise FileNotFoundError(f"{name} file was not found: {path}")

# Load the original data splits
train_df = pd.read_csv(data_paths["train"], low_memory=False)
val_df = pd.read_csv(data_paths["validation"], low_memory=False)
test_df = pd.read_csv(data_paths["test"], low_memory=False)

print("scikit-learn version:", sklearn.__version__)
print()
print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

print("\nAvailable columns:")
print(train_df.columns.tolist())

print("\nTarget distribution in Train:")
print(train_df["is_late"].value_counts(normalize=True).sort_index())

scikit-learn version: 1.6.1

Train shape: (67534, 20)
Validation shape: (14472, 20)
Test shape: (14472, 20)

Available columns:
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'total_price', 'total_freight', 'num_items', 'num_unique_sellers', 'total_payment', 'max_installments', 'main_payment_type', 'is_late']

Target distribution in Train:
is_late
0    0.909734
1    0.090266
Name: proportion, dtype: float64


In [3]:
FEATURE_COLUMNS = [
    "customer_state",
    "total_price",
    "total_freight",
    "num_items",
    "num_unique_sellers",
    "total_payment",
    "max_installments",
    "main_payment_type",
    "purchase_year",
    "purchase_month",
    "purchase_dayofweek",
    "purchase_hour",
    "is_weekend",
    "estimated_delivery_days",
    "freight_ratio",
]

TARGET_COLUMN = "is_late"

REQUIRED_RAW_COLUMNS = [
    "customer_state",
    "total_price",
    "total_freight",
    "num_items",
    "num_unique_sellers",
    "total_payment",
    "max_installments",
    "main_payment_type",
    "order_purchase_timestamp",
    "order_estimated_delivery_date",
    TARGET_COLUMN,
]


def build_features(df):
    """Create model features without fitting encoders or imputers."""
    missing_columns = [
        column for column in REQUIRED_RAW_COLUMNS
        if column not in df.columns
    ]

    if missing_columns:
        raise ValueError(f"Missing required columns: {missing_columns}")

    features = df.copy()

    features["order_purchase_timestamp"] = pd.to_datetime(
        features["order_purchase_timestamp"],
        errors="coerce",
    )
    features["order_estimated_delivery_date"] = pd.to_datetime(
        features["order_estimated_delivery_date"],
        errors="coerce",
    )

    features["purchase_year"] = features["order_purchase_timestamp"].dt.year
    features["purchase_month"] = features["order_purchase_timestamp"].dt.month
    features["purchase_dayofweek"] = (
        features["order_purchase_timestamp"].dt.dayofweek
    )
    features["purchase_hour"] = features["order_purchase_timestamp"].dt.hour

    features["is_weekend"] = (
        features["purchase_dayofweek"].isin([5, 6]).astype(int)
    )

    features["estimated_delivery_days"] = (
        features["order_estimated_delivery_date"]
        - features["order_purchase_timestamp"]
    ).dt.total_seconds() / (24 * 60 * 60)

    features["freight_ratio"] = (
        features["total_freight"]
        / (features["total_price"] + 0.001)
    )

    return features[FEATURE_COLUMNS]


X_train = build_features(train_df)
X_val = build_features(val_df)
X_test = build_features(test_df)

y_train = train_df[TARGET_COLUMN].astype(int)
y_val = val_df[TARGET_COLUMN].astype(int)
y_test = test_df[TARGET_COLUMN].astype(int)

print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)
print("X_test shape:", X_test.shape)

print("\nFeature data types:")
print(X_train.dtypes)

print("\nMissing values in Train:")
print(X_train.isna().sum())

X_train shape: (67534, 15)
X_val shape: (14472, 15)
X_test shape: (14472, 15)

Feature data types:
customer_state              object
total_price                float64
total_freight              float64
num_items                  float64
num_unique_sellers         float64
total_payment              float64
max_installments           float64
main_payment_type           object
purchase_year                int32
purchase_month               int32
purchase_dayofweek           int32
purchase_hour                int32
is_weekend                   int64
estimated_delivery_days    float64
freight_ratio              float64
dtype: object

Missing values in Train:
customer_state             0
total_price                0
total_freight              0
num_items                  0
num_unique_sellers         0
total_payment              1
max_installments           1
main_payment_type          1
purchase_year              0
purchase_month             0
purchase_dayofweek         0
purchase_hour    

In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

CATEGORICAL_COLUMNS = [
    "customer_state",
    "main_payment_type",
]

NUMERICAL_COLUMNS = [
    column
    for column in FEATURE_COLUMNS
    if column not in CATEGORICAL_COLUMNS
]

numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False,
            ),
        ),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, NUMERICAL_COLUMNS),
        ("categorical", categorical_pipeline, CATEGORICAL_COLUMNS),
    ],
    remainder="drop",
)

print("Categorical columns:")
print(CATEGORICAL_COLUMNS)

print("\nNumerical columns:")
print(NUMERICAL_COLUMNS)

print("\nPreprocessor created successfully.")

Categorical columns:
['customer_state', 'main_payment_type']

Numerical columns:
['total_price', 'total_freight', 'num_items', 'num_unique_sellers', 'total_payment', 'max_installments', 'purchase_year', 'purchase_month', 'purchase_dayofweek', 'purchase_hour', 'is_weekend', 'estimated_delivery_days', 'freight_ratio']

Preprocessor created successfully.


In [5]:
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    HistGradientBoostingClassifier,
)

RANDOM_STATE = 42

model_candidates = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=250,
        max_depth=16,
        min_samples_leaf=3,
        class_weight="balanced_subsample",
        n_jobs=-1,
        random_state=RANDOM_STATE,
    ),

    "Hist Gradient Boosting": HistGradientBoostingClassifier(
        max_iter=200,
        learning_rate=0.08,
        max_leaf_nodes=31,
        l2_regularization=1.0,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    ),
}

model_pipelines = {
    model_name: Pipeline(
        steps=[
            ("preprocessor", clone(preprocessor)),
            ("classifier", model),
        ]
    )
    for model_name, model in model_candidates.items()
}

print("Models prepared for comparison:")

for model_name in model_pipelines:
    print("-", model_name)

Models prepared for comparison:
- Logistic Regression
- Random Forest
- Hist Gradient Boosting


In [6]:
import time

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
)

validation_results = []
trained_models = {}

for model_name, pipeline in model_pipelines.items():
    print(f"Training: {model_name}...")

    start_time = time.perf_counter()

    pipeline.fit(X_train, y_train)

    training_time = time.perf_counter() - start_time

    val_predictions = pipeline.predict(X_val)
    val_probabilities = pipeline.predict_proba(X_val)[:, 1]

    model_result = {
        "model": model_name,
        "accuracy": accuracy_score(y_val, val_predictions),
        "precision": precision_score(
            y_val,
            val_predictions,
            zero_division=0,
        ),
        "recall": recall_score(
            y_val,
            val_predictions,
            zero_division=0,
        ),
        "f1_score": f1_score(
            y_val,
            val_predictions,
            zero_division=0,
        ),
        "roc_auc": roc_auc_score(
            y_val,
            val_probabilities,
        ),
        "pr_auc": average_precision_score(
            y_val,
            val_probabilities,
        ),
        "training_seconds": training_time,
    }

    validation_results.append(model_result)
    trained_models[model_name] = pipeline

    print(f"Completed in {training_time:.2f} seconds.\n")

validation_results_df = (
    pd.DataFrame(validation_results)
    .sort_values(
        by=["pr_auc", "f1_score"],
        ascending=False,
    )
    .reset_index(drop=True)
)

print("Validation comparison:")
display(validation_results_df.round(4))

Training: Logistic Regression...
Completed in 2.26 seconds.

Training: Random Forest...
Completed in 16.13 seconds.

Training: Hist Gradient Boosting...
Completed in 9.08 seconds.

Validation comparison:


,model,accuracy,precision,recall,f1_score,roc_auc,pr_auc,training_seconds
0,Random Forest,0.9144,0.2206,0.2380,0.2290,0.7023,0.1569,16.1291
1,Hist Gradient Boosting,0.8396,0.1532,0.4424,0.2276,0.7246,0.1558,9.0824
2,Logistic Regression,0.2644,0.0664,0.9780,0.1244,0.7380,0.1300,2.2589


In [7]:
from sklearn.metrics import precision_recall_curve

threshold_results = []
best_thresholds = {}

for model_name, pipeline in trained_models.items():
    val_probabilities = pipeline.predict_proba(X_val)[:, 1]

    precision_values, recall_values, thresholds = precision_recall_curve(
        y_val,
        val_probabilities,
    )

    f1_values = (
        2 * precision_values[:-1] * recall_values[:-1]
        / (
            precision_values[:-1]
            + recall_values[:-1]
            + 1e-12
        )
    )

    best_index = np.argmax(f1_values)
    best_threshold = float(thresholds[best_index])

    tuned_predictions = (
        val_probabilities >= best_threshold
    ).astype(int)

    best_thresholds[model_name] = best_threshold

    threshold_results.append(
        {
            "model": model_name,
            "best_threshold": best_threshold,
            "accuracy": accuracy_score(
                y_val,
                tuned_predictions,
            ),
            "precision": precision_score(
                y_val,
                tuned_predictions,
                zero_division=0,
            ),
            "recall": recall_score(
                y_val,
                tuned_predictions,
                zero_division=0,
            ),
            "f1_score": f1_score(
                y_val,
                tuned_predictions,
                zero_division=0,
            ),
            "roc_auc": roc_auc_score(
                y_val,
                val_probabilities,
            ),
            "pr_auc": average_precision_score(
                y_val,
                val_probabilities,
            ),
        }
    )

threshold_results_df = (
    pd.DataFrame(threshold_results)
    .sort_values(
        by=["f1_score", "pr_auc"],
        ascending=False,
    )
    .reset_index(drop=True)
)

print("Validation results after threshold tuning:")
display(threshold_results_df.round(4))

Validation results after threshold tuning:


,model,best_threshold,accuracy,precision,recall,f1_score,roc_auc,pr_auc
0,Hist Gradient Boosting,0.5793,0.8886,0.1886,0.3286,0.2396,0.7246,0.1558
1,Random Forest,0.4981,0.9132,0.2189,0.2432,0.2304,0.7023,0.1569
2,Logistic Regression,0.7889,0.8727,0.1661,0.3441,0.2241,0.7380,0.1300


In [8]:
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
)

SELECTED_MODEL_NAME = threshold_results_df.iloc[0]["model"]
SELECTED_THRESHOLD = best_thresholds[SELECTED_MODEL_NAME]

print("Selected model:", SELECTED_MODEL_NAME)
print("Selected threshold:", round(SELECTED_THRESHOLD, 4))

# Combine Train and Validation after model selection
X_train_final = pd.concat(
    [X_train, X_val],
    ignore_index=True,
)

y_train_final = pd.concat(
    [y_train, y_val],
    ignore_index=True,
)

# Create and train the final model
final_model = clone(model_pipelines[SELECTED_MODEL_NAME])

final_model.fit(
    X_train_final,
    y_train_final,
)

# Final evaluation on untouched Test data
test_probabilities = final_model.predict_proba(X_test)[:, 1]

test_predictions = (
    test_probabilities >= SELECTED_THRESHOLD
).astype(int)

final_test_results = {
    "model": SELECTED_MODEL_NAME,
    "threshold": SELECTED_THRESHOLD,
    "accuracy": accuracy_score(
        y_test,
        test_predictions,
    ),
    "precision": precision_score(
        y_test,
        test_predictions,
        zero_division=0,
    ),
    "recall": recall_score(
        y_test,
        test_predictions,
        zero_division=0,
    ),
    "f1_score": f1_score(
        y_test,
        test_predictions,
        zero_division=0,
    ),
    "roc_auc": roc_auc_score(
        y_test,
        test_probabilities,
    ),
    "pr_auc": average_precision_score(
        y_test,
        test_probabilities,
    ),
}

print("\nFinal Test Results:")

for metric_name, metric_value in final_test_results.items():
    if isinstance(metric_value, float):
        print(f"{metric_name}: {metric_value:.4f}")
    else:
        print(f"{metric_name}: {metric_value}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, test_predictions))

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        test_predictions,
        digits=4,
        zero_division=0,
    )
)

Selected model: Hist Gradient Boosting
Selected threshold: 0.5793

Final Test Results:
model: Hist Gradient Boosting
threshold: 0.5793
accuracy: 0.9120
precision: 0.0928
recall: 0.0376
f1_score: 0.0535
roc_auc: 0.6366
pr_auc: 0.0921

Confusion Matrix:
[[13163   352]
 [  921    36]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9346    0.9740    0.9539     13515
           1     0.0928    0.0376    0.0535       957

    accuracy                         0.9120     14472
   macro avg     0.5137    0.5058    0.5037     14472
weighted avg     0.8789    0.9120    0.8943     14472



In [9]:
# Evaluate the exact model used during validation threshold tuning
selected_validation_model = model_pipelines[SELECTED_MODEL_NAME]

test_probabilities_original = selected_validation_model.predict_proba(X_test)[:, 1]
test_predictions_original = (
    test_probabilities_original >= SELECTED_THRESHOLD
).astype(int)

corrected_test_results = {
    "model": SELECTED_MODEL_NAME,
    "threshold": SELECTED_THRESHOLD,
    "accuracy": accuracy_score(y_test, test_predictions_original),
    "precision": precision_score(
        y_test, test_predictions_original, zero_division=0
    ),
    "recall": recall_score(
        y_test, test_predictions_original, zero_division=0
    ),
    "f1_score": f1_score(
        y_test, test_predictions_original, zero_division=0
    ),
    "roc_auc": roc_auc_score(y_test, test_probabilities_original),
    "pr_auc": average_precision_score(
        y_test, test_probabilities_original
    ),
}

print("Corrected Test Results:")
for metric_name, metric_value in corrected_test_results.items():
    if isinstance(metric_value, float):
        print(f"{metric_name}: {metric_value:.4f}")
    else:
        print(f"{metric_name}: {metric_value}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, test_predictions_original))

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        test_predictions_original,
        digits=4,
        zero_division=0,
    )
)

Corrected Test Results:
model: Hist Gradient Boosting
threshold: 0.5793
accuracy: 0.7720
precision: 0.0644
recall: 0.1808
f1_score: 0.0949
roc_auc: 0.5235
pr_auc: 0.0739

Confusion Matrix:
[[11000  2515]
 [  784   173]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9335    0.8139    0.8696     13515
           1     0.0644    0.1808    0.0949       957

    accuracy                         0.7720     14472
   macro avg     0.4989    0.4973    0.4823     14472
weighted avg     0.8760    0.7720    0.8184     14472



In [10]:
def monthly_late_summary(df, split_name):
    temp = df.copy()
    temp["order_purchase_timestamp"] = pd.to_datetime(
        temp["order_purchase_timestamp"]
    )
    temp["year_month"] = (
        temp["order_purchase_timestamp"]
        .dt.to_period("M")
        .astype(str)
    )

    summary = (
        temp.groupby("year_month")
        .agg(
            orders=("is_late", "size"),
            late_orders=("is_late", "sum"),
            late_ratio=("is_late", "mean"),
        )
        .reset_index()
    )

    summary["late_ratio"] = summary["late_ratio"] * 100
    summary["split"] = split_name
    return summary


monthly_summary = pd.concat(
    [
        monthly_late_summary(train_df, "Train"),
        monthly_late_summary(val_df, "Validation"),
        monthly_late_summary(test_df, "Test"),
    ],
    ignore_index=True,
)

print("Monthly late-order distribution:")
display(monthly_summary.tail(15).round(2))

Monthly late-order distribution:


,year_month,orders,late_orders,late_ratio,split
10,2017-08,4193,139,3.32,Train
11,2017-09,4150,216,5.20,Train
12,2017-10,4478,237,5.29,Train
13,2017-11,7289,1043,14.31,Train
14,2017-12,5513,462,8.38,Train
15,2018-01,7069,464,6.56,Train
16,2018-02,6555,1048,15.99,Train
17,2018-03,7003,1496,21.36,Train
18,2018-04,3212,206,6.41,Train
19,2018-04,3586,155,4.32,Validation


In [11]:
from sklearn.model_selection import TimeSeriesSplit

# Development data only — Test data is not used here
X_development = pd.concat(
    [X_train, X_val],
    ignore_index=True
)

y_development = pd.concat(
    [y_train, y_val],
    ignore_index=True
)

time_cv = TimeSeriesSplit(n_splits=4)
cv_results = []

for model_name in model_pipelines:
    print(f"\nEvaluating: {model_name}")

    for fold_number, (train_indices, val_indices) in enumerate(
        time_cv.split(X_development),
        start=1
    ):
        fold_model = clone(model_pipelines[model_name])

        X_fold_train = X_development.iloc[train_indices]
        y_fold_train = y_development.iloc[train_indices]

        X_fold_val = X_development.iloc[val_indices]
        y_fold_val = y_development.iloc[val_indices]

        fold_model.fit(X_fold_train, y_fold_train)

        fold_probabilities = fold_model.predict_proba(X_fold_val)[:, 1]
        fold_predictions = (fold_probabilities >= 0.5).astype(int)

        fold_result = {
            "model": model_name,
            "fold": fold_number,
            "roc_auc": roc_auc_score(
                y_fold_val,
                fold_probabilities
            ),
            "pr_auc": average_precision_score(
                y_fold_val,
                fold_probabilities
            ),
            "f1_score": f1_score(
                y_fold_val,
                fold_predictions,
                zero_division=0
            ),
            "recall": recall_score(
                y_fold_val,
                fold_predictions,
                zero_division=0
            ),
        }

        cv_results.append(fold_result)

        print(
            f"Fold {fold_number}: "
            f"ROC-AUC={fold_result['roc_auc']:.4f}, "
            f"PR-AUC={fold_result['pr_auc']:.4f}"
        )

cv_results_df = pd.DataFrame(cv_results)

cv_summary = (
    cv_results_df
    .groupby("model")
    .agg(
        roc_auc_mean=("roc_auc", "mean"),
        roc_auc_std=("roc_auc", "std"),
        pr_auc_mean=("pr_auc", "mean"),
        pr_auc_std=("pr_auc", "std"),
        f1_mean=("f1_score", "mean"),
        recall_mean=("recall", "mean"),
    )
    .sort_values(
        by=["pr_auc_mean", "roc_auc_mean"],
        ascending=False
    )
    .reset_index()
)

print("\nTime-Series Cross-Validation Summary:")
display(cv_summary.round(4))


Evaluating: Logistic Regression
Fold 1: ROC-AUC=0.6492, PR-AUC=0.0915
Fold 2: ROC-AUC=0.6632, PR-AUC=0.1783
Fold 3: ROC-AUC=0.6799, PR-AUC=0.3093
Fold 4: ROC-AUC=0.7421, PR-AUC=0.1309

Evaluating: Random Forest
Fold 1: ROC-AUC=0.6188, PR-AUC=0.0752
Fold 2: ROC-AUC=0.6473, PR-AUC=0.1865
Fold 3: ROC-AUC=0.6677, PR-AUC=0.2855
Fold 4: ROC-AUC=0.7011, PR-AUC=0.1431

Evaluating: Hist Gradient Boosting
Fold 1: ROC-AUC=0.6195, PR-AUC=0.0723
Fold 2: ROC-AUC=0.6376, PR-AUC=0.1727
Fold 3: ROC-AUC=0.6574, PR-AUC=0.2653
Fold 4: ROC-AUC=0.7039, PR-AUC=0.1390

Time-Series Cross-Validation Summary:


,model,roc_auc_mean,roc_auc_std,pr_auc_mean,pr_auc_std,f1_mean,recall_mean
0,Logistic Regression,0.6836,0.0410,0.1775,0.0947,0.2127,0.5620
1,Random Forest,0.6587,0.0346,0.1726,0.0881,0.1122,0.1168
2,Hist Gradient Boosting,0.6546,0.0363,0.1623,0.0804,0.2294,0.4740


In [12]:
from sklearn.metrics import precision_recall_curve
import numpy as np

SELECTED_MODEL_NAME = "Logistic Regression"

oof_true_values = []
oof_probabilities = []

# Generate out-of-fold probabilities using chronological folds
for fold_number, (train_indices, val_indices) in enumerate(
    time_cv.split(X_development),
    start=1
):
    fold_model = clone(model_pipelines[SELECTED_MODEL_NAME])

    X_fold_train = X_development.iloc[train_indices]
    y_fold_train = y_development.iloc[train_indices]

    X_fold_val = X_development.iloc[val_indices]
    y_fold_val = y_development.iloc[val_indices]

    fold_model.fit(X_fold_train, y_fold_train)

    fold_probabilities = fold_model.predict_proba(X_fold_val)[:, 1]

    oof_true_values.extend(y_fold_val.to_numpy())
    oof_probabilities.extend(fold_probabilities)

oof_true_values = np.asarray(oof_true_values)
oof_probabilities = np.asarray(oof_probabilities)

precision_values, recall_values, threshold_values = (
    precision_recall_curve(
        oof_true_values,
        oof_probabilities
    )
)

f1_values = (
    2 * precision_values[:-1] * recall_values[:-1]
    / (
        precision_values[:-1]
        + recall_values[:-1]
        + 1e-10
    )
)

best_index = np.argmax(f1_values)
FINAL_THRESHOLD = threshold_values[best_index]

oof_predictions = (
    oof_probabilities >= FINAL_THRESHOLD
).astype(int)

print("Selected model:", SELECTED_MODEL_NAME)
print(f"Selected threshold: {FINAL_THRESHOLD:.4f}")

print("\nCross-validation metrics at selected threshold:")
print(
    f"Precision: "
    f"{precision_score(oof_true_values, oof_predictions, zero_division=0):.4f}"
)
print(
    f"Recall: "
    f"{recall_score(oof_true_values, oof_predictions, zero_division=0):.4f}"
)
print(
    f"F1-score: "
    f"{f1_score(oof_true_values, oof_predictions, zero_division=0):.4f}"
)
print(
    f"ROC-AUC: "
    f"{roc_auc_score(oof_true_values, oof_probabilities):.4f}"
)
print(
    f"PR-AUC: "
    f"{average_precision_score(oof_true_values, oof_probabilities):.4f}"
)

print("\nConfusion Matrix:")
print(confusion_matrix(oof_true_values, oof_predictions))

Selected model: Logistic Regression
Selected threshold: 0.6097

Cross-validation metrics at selected threshold:
Precision: 0.1602
Recall: 0.3830
F1-score: 0.2259
ROC-AUC: 0.6406
PR-AUC: 0.1478

Confusion Matrix:
[[47143 12323]
 [ 3787  2351]]


In [13]:
# Train the final selected pipeline on Train + Validation
final_model = clone(model_pipelines[SELECTED_MODEL_NAME])

final_model.fit(
    X_development,
    y_development
)

# Final evaluation on the untouched chronological Test set
final_test_probabilities = final_model.predict_proba(X_test)[:, 1]

final_test_predictions = (
    final_test_probabilities >= FINAL_THRESHOLD
).astype(int)

final_test_metrics = {
    "model": SELECTED_MODEL_NAME,
    "threshold": FINAL_THRESHOLD,
    "accuracy": accuracy_score(
        y_test,
        final_test_predictions
    ),
    "precision": precision_score(
        y_test,
        final_test_predictions,
        zero_division=0
    ),
    "recall": recall_score(
        y_test,
        final_test_predictions,
        zero_division=0
    ),
    "f1_score": f1_score(
        y_test,
        final_test_predictions,
        zero_division=0
    ),
    "roc_auc": roc_auc_score(
        y_test,
        final_test_probabilities
    ),
    "pr_auc": average_precision_score(
        y_test,
        final_test_probabilities
    ),
}

print("Final chronological Test results:")

for metric_name, metric_value in final_test_metrics.items():
    if isinstance(metric_value, float):
        print(f"{metric_name}: {metric_value:.4f}")
    else:
        print(f"{metric_name}: {metric_value}")

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_test,
        final_test_predictions
    )
)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        final_test_predictions,
        digits=4,
        zero_division=0
    )
)

Final chronological Test results:
model: Logistic Regression
threshold: 0.6097
accuracy: 0.4654
precision: 0.0985
recall: 0.8694
f1_score: 0.1770
roc_auc: 0.6864
pr_auc: 0.1251

Confusion Matrix:
[[5903 7612]
 [ 125  832]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9793    0.4368    0.6041     13515
           1     0.0985    0.8694    0.1770       957

    accuracy                         0.4654     14472
   macro avg     0.5389    0.6531    0.3906     14472
weighted avg     0.9210    0.4654    0.5759     14472



In [14]:
import json
import joblib
import sklearn

from datetime import datetime, timezone
from pathlib import Path

artifact_directory = Path("artifacts")
artifact_directory.mkdir(parents=True, exist_ok=True)

model_output_path = (
    artifact_directory
    / "delivery_delay_pipeline_v1.pkl"
)

features_output_path = (
    artifact_directory
    / "feature_columns_v1.pkl"
)

metadata_output_path = (
    artifact_directory
    / "delivery_delay_model_metadata_v1.json"
)

# Save the complete preprocessing + model pipeline
joblib.dump(
    final_model,
    model_output_path
)

# Save the required input feature names
joblib.dump(
    FEATURE_COLUMNS,
    features_output_path
)

serializable_test_metrics = {}

for metric_name, metric_value in final_test_metrics.items():
    if isinstance(metric_value, (float, np.floating)):
        serializable_test_metrics[metric_name] = float(metric_value)
    else:
        serializable_test_metrics[metric_name] = metric_value

model_metadata = {
    "model_name": SELECTED_MODEL_NAME,
    "model_version": "v1.0",
    "prediction_task": "delivery_delay_classification",
    "target_column": TARGET_COLUMN,
    "decision_threshold": float(FINAL_THRESHOLD),
    "feature_columns": FEATURE_COLUMNS,
    "test_metrics": serializable_test_metrics,
    "training_rows": int(len(X_development)),
    "test_rows": int(len(X_test)),
    "scikit_learn_version": sklearn.__version__,
    "trained_at_utc": datetime.now(timezone.utc).isoformat(),
    "known_limitation": (
        "Temporal concept drift and class imbalance "
        "reduce precision across future months."
    ),
}

with open(
    metadata_output_path,
    "w",
    encoding="utf-8"
) as metadata_file:
    json.dump(
        model_metadata,
        metadata_file,
        indent=4,
        ensure_ascii=False
    )

print("Artifacts saved successfully:")
print("Model:", model_output_path.resolve())
print("Features:", features_output_path.resolve())
print("Metadata:", metadata_output_path.resolve())

Artifacts saved successfully:
Model: C:\Users\sa\Desktop\MLOps-Qafza-2026\Tasks\Task-02\artifacts\delivery_delay_pipeline_v1.pkl
Features: C:\Users\sa\Desktop\MLOps-Qafza-2026\Tasks\Task-02\artifacts\feature_columns_v1.pkl
Metadata: C:\Users\sa\Desktop\MLOps-Qafza-2026\Tasks\Task-02\artifacts\delivery_delay_model_metadata_v1.json
